To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm
# Install latest Hugging Face for Gemma-3!
!pip install --no-deps git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [1]:
import os

os.environ["UNSLOTH_IS_PRESENT"] = "1"
from unsloth_zoo import loss_utils

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [ ]:
WANDB_API_KEY = "paste_your_wandb_api_key_here"
HUGGINGFACE_API_KEY = "paste_your_huggingface_api_key_here"
experiment_name = "gemma-3-4b-full-training"

import wandb

wandb.login(key=WANDB_API_KEY)
wandb.init(project="nlp-phonetic", name=experiment_name, id="8zbkjeq2", resume="allow")

### Get new phonetic token, training and valida data

In [2]:
import json

def get_data_from_json(file_path):
  with open(file_path, "r", encoding="utf-8") as f:
      data = json.load(f)
  return data

phonetic_token = get_data_from_json("./data/phonetic_token.json")
training_data = get_data_from_json("./data/training_data.json")
validation_data = get_data_from_json("./data/valid_data.json")
test_data_input = get_data_from_json("./data/test_inputs.json")

len(training_data)

9970

### Unsloth

In [3]:
# from unsloth import FastLanguageModel
# import torch
# max_seq_length = 1024 # Choose any! We auto support RoPE Scaling internally!
# dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+

# # 4bit pre quantized models we support for 4x faster downloading + no OOMs.
# fourbit_models = [
#     "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
#     "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
#     "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
#     "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
#     "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
#     "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
#     "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
#     "unsloth/Phi-3-medium-4k-instruct",
#     "unsloth/gemma-2-9b-bnb-4bit",
#     "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

#     "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
#     "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
#     "unsloth/Llama-3.2-3B-bnb-4bit",
#     "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

#     "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
# ] # More models at https://huggingface.co/unsloth

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/gemma-3-4b-pt", # YOUR MODEL YOU USED FOR TRAINING
#     max_seq_length = max_seq_length,
#     dtype = dtype,
#     load_in_4bit=False,
#     load_in_8bit=False,
#     full_finetuning=True,
#     token = HUGGINGFACE_API_KEY, # use one if using gated models like meta-llama/Llama-2-7b-hf,
# )

In [8]:
from unsloth import FastModel
import torch
max_seq_length = 1024

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    "unsloth/gemma-3-4b-pt-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

_, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-pt-unsloth-bnb-4bit",
    # model_name = "Pongsaky/gemma-3-4b-it-unsloth-bnb-4bit-phonetic-with-token-added-1-epoches",
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    token = HUGGINGFACE_API_KEY, # use one if using gated models like meta-llama/Llama-2-7b-hf,
)

==((====))==  Unsloth 2025.3.19: Fast Gemma3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.673 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [9]:
# Gemma-3 version ( Multi-modal )
old_model_vocab_size = model.config.text_config.vocab_size

# old_model_vocab_size = model.config.vocab_size
old_model_vocab_size

NameError: name 'model' is not defined

In [ ]:
# Gemma-3 version ( Mulit-modal )
old_tokenizer_vocab_size = len(tokenizer.tokenizer.vocab)

# old_tokenizer_vocab_size = len(tokenizer)
old_tokenizer_vocab_size

In [12]:
from collections import defaultdict
import re

def extract_phonetic_combinations(training_data, tokenizer):
    # Define regex pattern to match <r>[X][Y]text</r> patterns
    pattern = r'<r>(.+?)</r>'

    tag_dict = defaultdict(set)

    # Process each text in the training data
    for text in training_data:
        # Find all matches of the pattern in the text
        matches = re.findall(pattern, text)

        for item in matches:
            # Extract all tags (e.g., [a], [w])
            tags = re.findall(r'\[.*?\]', item)
            # Extract the word by removing tags
            word = re.sub(r'\[.*?\]', '', item).strip()
            # Add the word to each tag's set
            for tag in tags:
                if "<r>" in word:
                    print(f"Warning: <r> tag found in word '{word}'. Skipping this entry.")
                    continue
                tag_dict[tag].add(word)

    new_tag_dict = defaultdict(set)
    for key, value in tag_dict.items():
        for word in value:
            tokenized_words = tokenizer.tokenizer.encode(word, add_special_tokens=False)
            # tokenized_words = tokenizer.encode(word, add_special_tokens=False)
            if "<r>" in tokenized_words:
                print(f"Value: {value}")
                print(f"Tokenized word: {word}")
                print(f"Tokenized word: {tokenized_words}")
            for tokenized_word in tokenized_words:
                new_tag_dict[key].add(tokenized_word)

    return new_tag_dict

tag_dict = extract_phonetic_combinations(training_data, tokenizer)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [14]:
def mean_surround_new_tag_token(model, tag_dict, eps=1e-16):
    """
    Llama-3 for eg has untrained vectors in the base model.
    These include <|eot_id|>, <|start_header_id|>, <|end_header_id|>
    We reset them to the mean of the rest of the tokens
    """
    # All Unsloth Zoo code licensed under LGPLv3
    embedding_matrix = model.get_input_embeddings ().weight.clone()
    lm_head_matrix   = model.get_output_embeddings().weight.clone()

    tag_embedding_dict = {}
    tag_lm_head_dict = {}

    # for tag, ids_values in tag_dict.items():
    #   tag_embedding = torch.zeros(embedding_matrix[0].shape, dtype=torch.float32).to(device)
    #   tag_lm_head = torch.zeros(lm_head_matrix[0].shape, dtype=torch.float32).to(device)
    #   for ids in ids_values:
    #     tag_embbeding = tag_embedding + embedding_matrix[ids]
    #     tag_lm_head = tag_lm_head + lm_head_matrix[ids]
    #   tag_embedding_dict[tag] = tag_embbeding / len(ids_values)
    #   tag_lm_head_dict[tag] = tag_lm_head / len(ids_values)

    for tag, ids_values in tag_dict.items():
      # properly accumulate all the embeddings
      tag_embedding = torch.zeros_like(embedding_matrix[0])
      tag_lm_head   = torch.zeros_like(lm_head_matrix[0])
      for idx in ids_values:
          tag_embedding += embedding_matrix[idx]
          tag_lm_head   += lm_head_matrix[idx]
      tag_embedding_dict[tag] = tag_embedding / len(ids_values)
      tag_lm_head_dict[tag] = tag_lm_head / len(ids_values)

    # Get untrained tokens
    # indicator_untrained = torch.amax(embedding_matrix, axis = 1) <= eps
    # where_untrained = torch.where(indicator_untrained)[0]
    # n_untrained = where_untrained.shape[0]
    # n_trained = embedding_matrix.shape[0] - n_untrained

    # if n_untrained != 0:
    #     print(
    #         f"Unsloth: Not an error, but your model has {n_untrained} untrained tokens.\n"\
    #         "We shall set them to the mean of the other trained tokens."
    #     )
    # pass

    # # Get sum of all items
    # sum_embedding = torch.sum(embedding_matrix, dtype = torch.float32, axis = 0)
    # sum_lm_head   = torch.sum(lm_head_matrix,   dtype = torch.float32, axis = 0)

    # # Remove bad tokens
    # sum_embedding -= torch.sum(embedding_matrix[where_untrained], dtype = torch.float32, axis = 0)
    # sum_lm_head   -= torch.sum(lm_head_matrix  [where_untrained], dtype = torch.float32, axis = 0)

    # # Find correct average by dividing by sum of trained tokens
    # mean_embedding = (sum_embedding / n_trained)
    # mean_lm_head   = (sum_lm_head   / n_trained)

    return tag_embedding_dict, tag_lm_head_dict

In [11]:
mean_surround_new_tag_token(model, tag_dict=tag_dict)[0]["[a]"].shape

torch.Size([2560])

In [47]:
import gc

# Clear deleted GPU items
for _ in range(3):
    gc.collect()
    torch.cuda.empty_cache()

In [15]:
def add_new_tokens(
    model,
    tokenizer,
    tag_dict,
    new_tokens = [],
    method = "mean",
    interpolation = 0.5,
):
    """
    Smartly resizes the tokenizer and adds new tokens to the model.
    We also disregard untrained tokens by removing them from the mean calculation.
    """
    # All Unsloth Zoo code licensed under LGPLv3
    assert(isinstance(new_tokens, (list, tuple)))
    assert(len(new_tokens) > 0)
    assert(method == "mean" or method == "interpolation")
    assert(interpolation >= 0 and interpolation <= 1)

    # Check if tokens already exist
    # Gemma-3 version
    overlapping_tokens = set(new_tokens) & set(tokenizer.tokenizer.vocab.keys())
    # overlapping_tokens = set(new_tokens) & set(tokenizer.vocab.keys())
    if len(overlapping_tokens) != 0:
        print(
            f"Unsloth: You're adding new_tokens = {new_tokens}\n"\
            f"There are tokens which are overlapping = {list(overlapping_tokens)}\n"\
            f"We shall safely ignore these overlapping tokens."
        )
        new_tokens = [x for x in new_tokens if x not in overlapping_tokens]

    # Get mean of trained tokens
    # mean_embedding, mean_lm_head = fix_untrained_tokens(model)

    # Weirdly be careful reserved tokens can pop out
    # mean_embedding, mean_lm_head = mean_of_trained_tokens(model)
    tag_embedding_dict, tag_lm_head_dict = mean_surround_new_tag_token(model, tag_dict)

    # Get old lengths
    old_input_embedding  = model.get_input_embeddings ().weight
    old_output_embedding = model.get_output_embeddings().weight
    old_input_length  = old_input_embedding .shape[0]
    old_output_length = old_output_embedding.shape[0]
    # Gemma-3 version ( Multi-modal )
    old_config_size   = model.config.text_config.vocab_size
    # old_config_size = model.config.vocab_size

    # Check for tied weights as well
    is_tied = (old_input_embedding.data_ptr() == old_output_embedding.data_ptr()) \
        or (model.config.tie_word_embeddings)

    # Add tokens!
    # Gemma-3 version
    old_length = len(tokenizer.tokenizer)
    # old_length = len(tokenizer)
    # Gemma-3 version
    tokenizer.tokenizer.add_tokens(new_tokens)
    # tokenizer.add_tokens(new_tokens)
    # Also resizes lm_head as well!
    # Gemma-3 version
    model.resize_token_embeddings(len(tokenizer.tokenizer))
    # model.resize_token_embeddings(len(tokenizer))

    # If we use interpolation, we interpolate between the mean embeddings and
    # the Word2Vec sum of the other vectors
    embedding_matrix = model.get_input_embeddings ().weight
    lm_head_matrix   = model.get_output_embeddings().weight

    # Confirm sizes are correct
    if embedding_matrix.shape[0] > (old_input_length  + len(new_tokens)):
        raise RuntimeError(
            "Unsloth: Embedding matrix size did not get resized properly. Please file a bug report!"
        )
    if lm_head_matrix.shape[0]  > (old_output_length + len(new_tokens)):
        raise RuntimeError(
            "Unsloth: LM Head matrix size did not get resized properly. Please file a bug report!"
        )
    if model.config.text_config.vocab_size  > (old_config_size   + len(new_tokens)):
        raise RuntimeError(
            "Unsloth: Model's config vocab_size did not get resized properly. Please file a bug report!"
        )
    pass

    key_list = list(tag_dict.keys())
    # Gemma-3 version
    key_ids_list = [tokenizer.tokenizer.encode(word ,add_special_tokens=False)[0] for word in key_list]
    # key_ids_list = [tokenizer.encode(word ,add_special_tokens=False)[0] for word in key_list]
    with torch.no_grad():
        for key, ids in zip(key_list, key_ids_list):
            tag_embedding = tag_embedding_dict[key]
            tag_lm_head = tag_lm_head_dict[key]
            embedding_matrix[ids] = tag_embedding
            lm_head_matrix[ids] = tag_lm_head
            pass

    # # Now set the new tokens to the mean!
    # embedding_matrix[old_length:] = mean_embedding
    # lm_head_matrix  [old_length:] = mean_lm_head
    # pass

    # We set a flag to say we need to train embeddings
    internal_model = model
    while hasattr(internal_model, "model"):
        internal_model._need_to_train_embeddings = True
        internal_model = internal_model.model
    pass
    internal_model._need_to_train_embeddings = True

    # Fix up all vocab sizes
    current_model = model
    while hasattr(current_model, "model") and hasattr(current_model, "config"):
        # Gemma-3 version
        if hasattr(current_model.config.text_config, "vocab_size"):
        # if hasattr(current_model.config, "vocab_size"):
            # Gemma-3 version
            current_model.config.text_config.update({"vocab_size" : len(tokenizer.tokenizer)})
            # current_model.config.update({"vocab_size" : len(tokenizer)})
        current_model = current_model.model
    if hasattr(current_model, "model") and hasattr(current_model, "config"):
        # Gemma-3 version
        if hasattr(current_model.config.text_config, "vocab_size"):
        # if hasattr(current_model.config, "vocab_size"):
            # Gemma-3 version
            current_model.config.text_config.update({"vocab_size" : len(tokenizer.tokenizer)})
            # current_model.config.update({"vocab_size" : len(tokenizer)})
    pass

    # Must tie lm_head and embed_tokens if they are tied!
    # Otherwise error will occur on saving models ie use save_model
    if is_tied: model.tie_weights()

    # Clear deleted GPU items
    for _ in range(3):
        gc.collect()
        torch.cuda.empty_cache()
    return

In [14]:
add_new_tokens(model, tokenizer, tag_dict, new_tokens=phonetic_token)

Unsloth: You're adding new_tokens = ['<klon8>', '<phonetic>', '<th>', '<r>', '</r>', '[0]', '[1]', '[2]', '[3]', '[4]', '[?]', '[@@]', '[N]', '[OO]', '[O]', '[UU]', '[UUa]', '[U]', '[a]', '[aa]', '[b]', '[c]', '[ch]', '[d]', '[e]', '[ee]', '[f]', '[h]', '[i]', '[ii]', '[iia]', '[j]', '[k]', '[kh]', '[khl]', '[khr]', '[khw]', '[kl]', '[kr]', '[kw]', '[l]', '[m]', '[n]', '[o]', '[oo]', '[p]', '[ph]', '[phl]', '[phr]', '[pl]', '[pr]', '[r]', '[s]', '[sr]', '[t]', '[th]', '[thr]', '[tr]', '[u]', '[uu]', '[uua]', '[w]', '[x]', '[xx]']
There are tokens which are overlapping = ['<th>']
We shall safely ignore these overlapping tokens.


In [15]:
new_model_vocab_size = model.config.text_config.vocab_size
# new_model_vocab_size = model.config.vocab_size
new_model_vocab_size

262208

In [16]:
new_tokenizer_vocab_size = len(tokenizer.tokenizer.vocab)
# new_tokenizer_vocab_size = len(tokenizer)
new_tokenizer_vocab_size

262208

In [17]:
text_example = training_data[0]
text_example

'<klon8> พอได้ยินเสียงระฆังข้างหลัง<r>[a][w]เขา</r>\tเห็นผู้<r>[a][w]เฒ่า</r>ออกจากชะวาก<r>[a]ผา</r>\nดูสรรพางค์ร่างกายแก่ช<r>[a]รา</r>\tแต่ผิว<r>[a]หน้า</r>นั้นละม้ายคล้ายทา<r>[o][k]รก</r>\nทรงเสื้อโขมพัสตรานุ่งผ้า<r>[a][w]ขาว</r>\tผมนั้น<r>[a][w]ยาว</r>ย้อยสยายประปราย<r>[o][k]ปรก</r>\nถือไม้เท้าเนาวรัตน์พัดขน<r>[o][k]นก</r>\tทำเดิน<r>[o][k]งก</r>งันมาแล้วพา<r>[i]ที</r>\nว่าดูราสามีนางผี<r>[Ua]เสื้อ</r>\tเป็นหน่อ<r>[Ua]เนื้อ</r>กษัตริย์ชาติราช<r>[i]สีห์</r>\nอย่าเผาศพนางยักษ์ด้วยอัค<r>[i]คี</r>\tภัยจะ<r>[i]มี</r>ถึงกายให้วาย<r>[a][n]ปราณ</r>\n'

In [18]:
print(tokenizer.tokenizer.tokenize(text_example))

['<klon8>', '▁พอ', 'ได้', 'ยิน', 'เสียง', 'ระ', 'ฆ', 'ัง', 'ข้าง', 'หลัง', '<r>', '[a]', '[w]', 'เขา', '</r>', '\t', 'เห็น', 'ผู้', '<r>', '[a]', '[w]', 'เ', 'ฒ', '่า', '</r>', 'ออกจาก', 'ช', 'ะ', 'ว', 'าก', '<r>', '[a]', 'ผ', 'า', '</r>', '\n', 'ดู', 'สรร', 'พ', 'าง', 'ค์', 'ร่างกาย', 'แก่', 'ช', '<r>', '[a]', 'รา', '</r>', '\t', 'แต่', 'ผิว', '<r>', '[a]', 'หน้า', '</r>', 'นั้น', 'ละ', 'ม', '้าย', 'คล', '้าย', 'ท', 'า', '<r>', '[o]', '[k]', 'รก', '</r>', '\n', 'ทรง', 'เสื้อ', 'โ', 'ข', 'ม', 'พ', 'ัส', 'ตรา', 'น', 'ุ่ง', 'ผ้า', '<r>', '[a]', '[w]', 'ขาว', '</r>', '\t', 'ผม', 'นั้น', '<r>', '[a]', '[w]', 'ยาว', '</r>', 'ย', '้อย', 'ส', 'ยาย', 'ประ', 'ป', 'ราย', '<r>', '[o]', '[k]', 'ป', 'รก', '</r>', '\n', 'ถือ', 'ไม้', 'เท้า', 'เ', 'นา', 'ว', 'รั', 'ต', 'น์', 'พ', 'ัด', 'ขน', '<r>', '[o]', '[k]', 'น', 'ก', '</r>', '\t', 'ทำ', 'เดิน', '<r>', '[o]', '[k]', 'ง', 'ก', '</r>', 'ง', 'ัน', 'มา', 'แล้ว', 'พา', '<r>', '[i]', 'ที', '</r>', '\n', 'ว่า', 'ดู', 'รา', 'สาม', 'ีน', 'าง', 'ผ', 'ี', '

We now add LoRA adapters so we only need to update a small amount of parameters!

In [19]:
# model = FastModel.get_peft_model(
#     model,
#     finetune_vision_layers     = False, # Turn off for just text!
#     finetune_language_layers   = True,  # Should leave on!
#     finetune_attention_modules = True,  # Attention good for GRPO
#     finetune_mlp_modules       = True,  # Should leave on always!

#     r = 8,           # Larger = higher accuracy, but might overfit
#     lora_alpha = 8,  # Recommended alpha == r at least
#     lora_dropout = 0,
#     bias = "none",
#     random_state = 3407,
# )

In [20]:
train_embeddins = True
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",]

if train_embeddins:
  target_modules = target_modules + ["lm_head", "embed_tokens"]

model = FastModel.get_peft_model(
    model,
    target_modules = target_modules,
    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:550: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['language_model.model.embed_tokens', 'language_model.lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


Unsloth: Making `base_model.model.vision_tower.vision_model` require gradients


In [21]:
for name, param in model.named_parameters():
  if any(substring in name for substring in ["lm_head", "embed_tokens"]):
    # param.requires_grad = True
    print(name, param.requires_grad)

base_model.model.language_model.model.embed_tokens.base_layer.weight False
base_model.model.language_model.model.embed_tokens.lora_embedding_A.default False
base_model.model.language_model.model.embed_tokens.lora_embedding_B.default False
base_model.model.language_model.lm_head.lora_A.default.weight True
base_model.model.language_model.lm_head.lora_B.default.weight True


<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [22]:
tokenizer.tokenizer.vocab["<eos>"]

1

In [23]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": training_data})
valid_dataset = Dataset.from_dict({"text": validation_data})

train_dataset = train_dataset.map(lambda x :  { "text" : x["text"] }).shuffle(seed=42)
valid_dataset = valid_dataset.map(lambda x :  { "text" : x["text"] }).shuffle(seed=42)

train_dataset, valid_dataset

Map:   0%|          | 0/9970 [00:00<?, ? examples/s]

Map:   0%|          | 0/1035 [00:00<?, ? examples/s]

(Dataset({
     features: ['text'],
     num_rows: 9970
 }),
 Dataset({
     features: ['text'],
     num_rows: 1035
 }))

In [24]:
train_dataset["text"][0]

'<klon8> ครานั้นท่านยายบุษ<r>[a]บา</r>\tปลอบลูกสาว<r>[a]ว่า</r>อย่าสะ<r>[U][n]อื้น</r>\nพ่อแม่ก็ได้สั่งไว้ยั่ง<r>[U][n]ยืน</r>\tหม่อม<r>[U][n]หมื่น</r>เธอก็รับปฏิ<r>[a][n]ญาณ</r>\nแต่ใจแม่นี้ยังกริ่งอยู่สิ่ง<r>[U][N]หนึ่ง</r>\tกลัวจะ<r>[U][N]หึง</r>กันวุ่นวายอายชาว<r>[a][n]บ้าน</r>\nอันเมียสองต้องห้ามตามโบ<r>[a][n]ราณ</r>\tเป็นกับใครก็รำคาญไม่เว้น<r>[o][n]คน</r>\nแม่สอนเจ้ามาแต่น้อยกว่าร้อย<r>[a][n]พัน</r>\tสุดสำ<r>[a][n]คัญ</r>แต่เพียงอดนั้นเป็น<r>[o][n]ต้น</r>\nอย่าทำชั่วเพราะว่าตัวของตัว<r>[o][n]จน</r>\tเขาเปรียบเทียบจงสู้ทนต้องเกรง<r>[ua]กลัว</r>\nใครจะด่าเจาะจังก็ชั่ง<r>[a][w]เขา</r>\tจงอด<r>[a][w]เอา</r>อย่าสำออยคอยฟ้อง<r>[ua]ผัว</r>\nอันคนดีนานดอกจึงออก<r>[ua]ตัว</r>\tถ้าคน<r>[ua]ชั่ว</r>เขาคงเห็นเป็นไป<r>[e][N]เอง</r>\n'

In [25]:
print(tokenizer.tokenizer.tokenize(train_dataset["text"][0]))

['<klon8>', '▁ค', 'รา', 'นั้น', 'ท่าน', 'ยาย', 'บุ', 'ษ', '<r>', '[a]', 'บา', '</r>', '\t', 'ปล', 'อบ', 'ลูก', 'สาว', '<r>', '[a]', 'ว่า', '</r>', 'อย่า', 'สะ', '<r>', '[U]', '[n]', 'อ', 'ื้น', '</r>', '\n', 'พ่อ', 'แม่', 'ก็ได้', 'สั่ง', 'ไว', '้ย', 'ั่ง', '<r>', '[U]', '[n]', 'ยืน', '</r>', '\t', 'หม', '่อม', '<r>', '[U]', '[n]', 'หม', 'ื่น', '</r>', 'เ', 'ธ', 'อก', '็', 'รับ', 'ปฏิ', '<r>', '[a]', '[n]', 'ญา', 'ณ', '</r>', '\n', 'แต่', 'ใจ', 'แม่', 'นี้', 'ยัง', 'ก', 'ริ', '่ง', 'อยู่', 'สิ่ง', '<r>', '[U]', '[N]', 'หนึ่ง', '</r>', '\t', 'กล', 'ัว', 'จะ', '<r>', '[U]', '[N]', 'ห', 'ึง', '</r>', 'กัน', 'ว', 'ุ่น', 'ว', 'าย', 'อ', 'าย', 'ชาว', '<r>', '[a]', '[n]', 'บ้าน', '</r>', '\n', 'อัน', 'เม', 'ีย', 'สอง', 'ต้อง', 'ห', '้าม', 'ตาม', 'โบ', '<r>', '[a]', '[n]', 'รา', 'ณ', '</r>', '\t', 'เป็น', 'กับ', 'ใคร', 'ก็', 'ร', 'ำ', 'คา', 'ญ', 'ไม่', 'เว', '้น', '<r>', '[o]', '[n]', 'คน', '</r>', '\n', 'แม่', 'สอน', 'เจ้า', 'มา', 'แต', '่น', '้อย', 'กว่า', 'ร', '้อย', '<r>', '[a]', '[n]', 'พ

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [26]:
from transformers import TrainerCallback
import torch

class TextOutputCallback(TrainerCallback):
    """
    A custom callback that generates and prints model outputs every `every_n_steps` steps.
    """
    def __init__(
        self,
        model,
        tokenizer,
        test_data_input: list[str],
        every_n_steps: int = 10,
        max_new_tokens: int = 468,
        temperature: float = 1,
        top_p: float = 0.95,
        top_k: int = 64
    ):
        """
        Args:
            model: The model to use for generation
            tokenizer: The tokenizer to use
            test_data_input (List[str]): List of input text for generation
            every_n_steps (int): Interval between generations
            max_new_tokens (int): Maximum tokens to generate
            temperature (float): Sampling temperature
            top_p (float): Nucleus sampling parameter
            top_k (int): Top-k sampling parameter
        """
        self.model = model
        self.tokenizer = tokenizer
        self.test_data_input = test_data_input
        self.every_n_steps = every_n_steps
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p
        self.top_k = top_k

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_n_steps == 0:
            print(f"\n=== Generating output at step {state.global_step} ===")

            rand_idx = int(torch.rand(1) * len(self.test_data_input))
            sample_text = self.test_data_input[rand_idx]

            # Prepare inputs
            inputs = self.tokenizer(
                [sample_text],
                return_tensors="pt",
                padding=True,
                truncation=True
            ).to(self.model.device)

            # Generate outputs
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    temperature=self.temperature,
                    top_p=self.top_p,
                    top_k=self.top_k,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            # Decode and print
            generated_text = self.tokenizer.decode(
                outputs[0],
                skip_special_tokens=False,
                clean_up_tokenization_spaces=True
            )

            print(f"Input: {sample_text}")
            print(f"Output: {generated_text}")
            print("=" * 80)

        return control

In [27]:
import numpy as np

epoch = 3
batch_size = 16
accumulation_steps = 1
number_time_text_output = 20
number_of_eval = 20
number_of_save_step_per_epoch = 1

total_steps = int(len(train_dataset) / (batch_size * accumulation_steps)) * epoch
warmup_steps = int(total_steps * 0.1)
every_n_steps = int(total_steps / number_time_text_output)
eval_steps = int(total_steps / number_of_eval)
save_steps = int(np.ceil((total_steps / ( number_of_save_step_per_epoch * epoch)) / eval_steps) * eval_steps)

In [28]:
total_steps, warmup_steps, every_n_steps, eval_steps, save_steps, experiment_name

(1869, 186, 93, 93, 651, 'gemma-3-4b-full-training')

In [29]:
from transformers import EarlyStoppingCallback

callbacks = [
    EarlyStoppingCallback(early_stopping_patience=5),
    TextOutputCallback(
        model=model,
        tokenizer=tokenizer,
        every_n_steps=every_n_steps,
        test_data_input=test_data_input
    )
]

In [30]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset= valid_dataset,
    dataset_text_field = "text",
    metric_for_best_model="eval_loss",
    eval_strategy='steps',
    save_strategy='steps',
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    callbacks = callbacks,
    args = TrainingArguments(
        per_device_train_batch_size = batch_size,
        gradient_accumulation_steps = accumulation_steps,
        warmup_steps = warmup_steps,
        num_train_epochs = epoch, # Set this for 1 full training run.,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir=f"./checkpoints-{experiment_name}",
        report_to = "wandb", # Use this for WandB etc,
        save_steps=save_steps,
        eval_steps=eval_steps,
        run_name=experiment_name,
        eval_strategy="steps",
        load_best_model_at_end=True,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/9970 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1035 [00:00<?, ? examples/s]

In [31]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<bos><klon8> ครั้นสรรพเสร็จเสด็จมาหน้าส<r>[a][m]นาม</r>\tพี่เลี้ยง<r>[a][m]ตาม</r>เคียงประคองทั้งสอง<r>[a][N]ข้าง</r>\nหนุ่มขนาดมหาดเล็กลูกขุน<r>[a][N]นาง</r>\tล้วนรูป<r>[a][N]ร่าง</r>รุ่นราวคราวพระ<r>[o][N]องค์</r>\nเชิญเครื่องอานพานพระศรีพระแสง<r>[e][t]เพชร</r>\tตาม<r>[e][t]เสด็จ</r>ยุรยาตรดังราช<r>[o][N]หงส์</r>\nให้ผูกสิงห์มิ่งม้ามังกร<r>[o][N]ทรง</r>\tไปรับ<r>[o][N]องค์</r>เชษฐาสุดสา<r>[O][n]คร</r>\nแล้วทรงนั่งหลังสิงห์กั้นกลิ้ง<r>[o][t]กลด</r>\tเผ่นพ<r>[o][t]ยศ</r>เยื้องไล่เช่นไกร<r>[O][n]สร</r>\nตำรวจเรียงเคียงข้างหนทาง<r>[O][n]จร</r>\tเข้าน<r>[O][n]คร</r>เสด็จมาถึงหน้า<r>[a][N]วัง</r>\nหยุดสิงห์ทรงตรงประตูเขารู้<r>[a][k]จัก</r>\tต่างถาม<r>[a][k]ทัก</r>ทุกคนเหมือนหน<r>[a][N]หลัง</r>\nพระเรียกหาฝรั่งเฝ้าเล่าให้<r>[a][N]ฟัง</r>\tเราออก<r>[a][N]นั่ง</r>หน้าพระลานชานชา<r>[a]ลา</r>\n'

In [32]:
tokenizer.decode(trainer.eval_dataset[100]["input_ids"])

'<bos><klon8> พระชื่นชอบตอบว่าบิดา<r>[i]นี้</r>\tแม้เท<r>[i]วี</r>ล่วงลับจะดับ<r>[u][n]สูญ</r>\nไม่ขออยู่ดูพักตร์ศักดิ์ตระ<r>[u][n]กูล</r>\tพระลูก<r>[u][n]ทูล</r>ด้วยเถิดหนาพ่อมา<r>[a][m]ตาม</r>\nแล้วมานั่งยังเกรินกระหนก<r>[o][t]รถ</r>\tแสนกำ<r>[o][t]สรด</r>เศร้าพระทัยจะใคร่<r>[a][m]ถาม</r>\nฝ่ายละเวงวัณฬาพะงา<r>[a][m]งาม</r>\tครั้น<r>[a][m]สาม</r>ยามเย็นเยียบเงียบสำ<r>[ia][N]เนียง</r>\nเสนาะดังจังหรีดวะหวีด<r>[x][w]แว่ว</r>\tเสียง<r>[x][w]แจ้ว</r>แจ้วไก่ขันสนั่น<r>[ia][N]เสียง</r>\nทุกก้านกิ่งมิ่งไม้เรไร<r>[ia][N]เรียง</r>\tชม้าย<r>[ia][N]เมียง</r>ดูพระอภัยม<r>[i]ณี</r>\nเห็นเศร้าสร้อยพลอยทุกข์จะปลุก<r>[U][m]ปลื้ม</r>\tให้หลง<r>[U][m]ลืม</r>ละเสน่ห์มเห<r>[i]สี</r>\nจึงเสแสร้งแกล้งว่าสุลา<r>[i]ลี</r>\tเวลา<r>[i]นี้</r>หนาวใจกระไร<r>[@][j]เลย</r>\n'

In [33]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA RTX A5000. Max memory = 23.673 GB.
7.199 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [34]:
trainer_stats = trainer.train()
wandb.finish()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,970 | Num Epochs = 3 | Total steps = 1,872
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 18,512,384/4,000,000,000 (0.46% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
93,3.466500,3.622817
186,2.653900,2.586152
279,2.136200,2.308631
372,2.008000,2.187206
465,1.939500,2.114798
558,2.108100,2.062776
651,1.961000,2.024424
744,1.822400,1.997192
837,1.840800,1.966003
930,1.844400,1.945313


Unsloth: Will smartly offload gradients to save VRAM!

=== Generating output at step 93 ===
Input: <klon8> จะรักน้อง
Output: <bos><klon8> จะรักน้องรักน้องกะจะจะจะจะรักนางรักนาง	ฉันทาพระราชนิเทศนาเสนาะให้ฟังได้ฟังได้ฟังได้ฟังหูได้หูได้หูได้หู	ฉันทาพระราชนิเทศนาเสนาะให้ฟังได้ฟังได้ฟังได้ฟังหูได้หูได้หูได้หู	นายพราหมณ์ทำท้วมงึกงักกึ่งเขินหวั่นหวั่นหวั่นหวั่น	พระองค์ยังไม่ทรงขรี่โกรธกว้างพ้องยื่นถอดอารมณ์คิดพินิจคิดพินิจคิดคิดตั้งพระเถรเสนา	นายพระราชนินทาคิดนึกในใจใจใจใจคิดว่าถ้าพี่จะหลอกนางจะได้ใจใจใจใจนางจะตามใจใจใจใจใจใจใจใจใจ	ก็ว่าอับหน้าใจใจใจใจใจให้หล่อราศรีใจใจใจใจใจใจใจใจใจใจ	ขอบใจพี่ว่าดีใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจ	ถอนใจให้เสร็จคิดวิหารามิราคมที่ใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจใจ

/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:220: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")



=== Generating output at step 744 ===
Input: <klon8> ถือพัดวาล
Output: <bos><klon8> ถือพัดวาลี่ใส่คีม<r>[o][m]หนม</r>	เป็นสอง<r>[o][m]สม</r>หวังให้พบ<r>[a]ฝา</r>
เข้าห้องนอนนัดกันให้ทัน<r>[a]มา</r>	จะ<r>[a]อา</r>ศรัยไม่ให้ใครเป็น<r>[ua]ตัว</r>
ฝ่ายพระนายแสนสุขสะบัด<r>[a][j]ไฟ</r>	คอยอยู่<r>[a][j]ใกล้</r>เตาแล้วมาประ<r>[ua]ชวร</r>
แสนดีใจจะได้เห็นกัน<r>[ua]วร</r>	จึง<r>[ua]พลอย</r>เปิดประตูดูกุ<r>[o][n]บน</r>
กุเวกอึ้งงงไม่รู้<r>[a]ว่า</r>	จึงอุ้ม<r>[a]มา</r>นึกว่ามีใคร<r>[o][n]ขน</r>
นี่พระนางนั่งหลับอยู่ทัน<r>[o][n]คน</r>	จึงอุ้ม<r>[o][n]ขึ้น</r>บนเตียงให้เคียง<r>[a][j]กาย</r>
ปลดกางเกงลงจากเกราะแตร<r>[a][t]ขวัญ</r>	ยัด<r>[a][t]หยัด</r>พับไว้ไม่ใกล้<r>[a][j]หลาย</r>
แล้วครางร้องรำพันเหมือนฟัน<r>[a][j]กราย</r>	แม่จะ<r>[a][j]ไป</r>ที่ไหนไปตาม<r>[a][n]กัน</r>
ข้าขวยเขินเมินมนต์ไม่ทัน<r>[a][j]ได้</r>	อ้อนวอน<r>[a][j]ใจ</r>พี่นางอย่าหึง<r>[a][n]หัน</r>
เจ้าพระยาชิดสนิทยิ่งนึก<r>[a][n]พลัน</r>	น้องรัก<r>[a][n]ฉัน</r>จึงข้ามจะตาม<r>[a][n]กัน</r>
เมื่อไรจะได้ผันผายกลับมา<r>[o][n]พ้น</r>	จะอย

/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:220: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")



=== Generating output at step 1395 ===
Input: <klon8> ขุนไกรได้
Output: <bos><klon8> ขุนไกรได้ฟังแกล้งว่า<r>[a][w]เจ้า</r>	ข้าขี้<r>[a][w]เบื่อ</r>จะมาหึงขี้คร้าน<r>[U][N]ยื่ง</r>
พูดเสียให้มันดูเอาผู้<r>[U][N]หญิง</r>	ที่เขา<r>[U][N]อึง</r>เอิบเอ๋ยเห็นเหงื่อ<r>[o]โซ</r>
มิใช่เด็กเป็นใหญ่ไม่ใจ<r>[a][N]จาง</r>	ก็เก่ง<r>[a][N]อย่าง</r>เจ้าไม่สู้ดูดี<r>[o]โหล</r>
พาลดุด่าทอดเสียไม่เชื่อ<r>[o]โอ้</r>	จะทูล<r>[o]โจ้</r>มิได้ก็ไม่ฟัง<r>[a][j]นาย</r>
จะช้าอยู่อย่างนี้ทำไม<r>[a][N]ช่าง</r>	ฤๅจะ<r>[a][N]ล้าง</r>เสียให้สิ้นบ้านพลัด<r>[a][j]หมาย</r>
ไปเป็นเกณฑ์เบื้องหน้าธา<r>[a][j]นัย</r>	ไม่<r>[a][j]ไว้</r>ให้เจ้าเป็นหงส์<r>[ia][N]เถียง</r>
ขืนอยู่ให้ลูกเมียมันป<r>ล้น</r>	กลัวพ่อจะ<r>[O]ส</r>ลอนทูลขอ<r>[ia][N]เสียง</r>
ถึงเจ้าจะขอไปก็ไม่<r>[ia][N]เกี่ยง</r>	เพราะพ่อ<r>[ia][N]เลี้ยง</r>เลี้ยงมาแต่<r>[a][j]ไร</r>
ถ้าไม่อยากผูกรักพากัน<r>[u][t]สูด</r>	จะหนี<r>[u][t]ขูด</r>ไปเชียงอินท์ให้จบ<r>[a][j]หาย</r>
มิฟังฟังจะเอาโทษถึงเค<r>[a][j]ราย</r>	อย่าหลง<r>[a][j]อาย</r>เลยเป็นมึงอย่าปลิด<r>[a][n]พลัน<

/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:220: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


eval/loss,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
eval/runtime,█▄▂▄▄▄▁▁▂▄▄▂▄▄▅▇▆▆▆▆
eval/samples_per_second,▁▅▆▅▄▅▇█▇▅▅▇▄▅▄▂▃▃▃▃
eval/steps_per_second,▁▅▆▅▄▅▇█▇▅▅▇▄▅▄▂▃▃▃▃
train/epoch,▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇██
train/global_step,▁▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇███
train/grad_norm,▁█▃▃▅▃▃▃▃▂▂▂▂▂▃▂▂▂▂▃▁▂▂▁▁▂▂▂▂▂▂▁▁▁▁▁▁▁▁▃
train/learning_rate,▁▁▃▃▆█████▇▇▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/loss,█▇▇▄▄▃▃▃▃▂▂▂▂▂▂▁▂▂▂▂▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,1.84015
eval/runtime,75.9292


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

7282.6121 seconds used for training.
121.38 minutes used for training.
Peak reserved memory = 10.811 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 73.34 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [37]:
# text = trainer.train_dataset['text'][0]
text = "<bos><klon8> พอได้ยินเสียงระฆัง"

outputs = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 468, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 512,
)

print("".join(tokenizer.batch_decode(outputs)))

<bos><klon8> พอได้ยินเสียงระฆังบังหลัง<r>[O][N]ห้อง</r>	เสียงโห่<r>[O][N]ร้อง</r>รบปีกจะยิก<r>[i][N]ยิง</r>
หน่อนรินทร์ยินเสียงให้เรียง<r>[i][N]วิ่ง</r>	เห็นท<r>[i][N]ลิ่ง</r>หนุนกันอยู่ชั้น<r>[ia][N]เวิ้ง</r>
เข้าประตูบูรีดูผู้<r>[a][t]พัศ</r>	เป็น<r>[a][t]รัด</r>ร้อยหมื่นประหม่า<r>[ia][N]เสียง</r>
เครื่องดนตรีปี่เลี้ยงต่างเสียง<r>[ia][N]เซิง</r>	พวกหญิง<r>[ia][N]เวียง</r>เข้ามาพลุ่งกระพุ่ง<r>[a][j]ไป</r>
หน่อกษัตริย์จัดพวกพหลพล<r>[a][k]หลัก</r>	เสด็จ<r>[a][k]จาก</r>สระน้อยเรียกผ<r>[a][j]ไพร่</r>
ยกพวกทูตสู้วิบัติปลิดพราย<r>[a][j]ตาย</r>	ต่างหลบ<r>[a][j]กาย</r>กอศิลาที่ขวา<r>[U]มือ</r>
เป็นเยี่ยงอย่างข้างกลองกระโดด<r>[a][N]หลัง</r>	เสียง<r>[a][N]ดัง</r>เฮฮ่าฮาทำหน้า<r>[U]อี๋</r>
บ้างเอาดาบวาบวิบัติตะโพก<r>[U]ซือ</r>	ทั้ง<r>[U]มือ</r>เล็บหักไว้มิได้<r>[o][N]พอง</r>
หน่อกษัตริย์ขับร่ามาหน้า<r>[a][n]ลาน</r>	พระหัส<r>[a][n]กัน</r>ทรงช้างอยู่ข้าง<r>[o][N]ขวง</r>
นัดหอกกลอกกอขวานออกฟาด<r>[o][N]ลง</r>	ขึ้นขี่<r>[o][N]ตรง</r>รุกรบสมทบ<r>[a][n]กัน</r>
คนลนลานช้างม้าเข้าย่า<r>[a][t]หยัด</r>	ถูก

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [38]:
from transformers import TextStreamer

text = "<klon8> คนธรรพ์ครั้น"

_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 512, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<bos><klon8> คนธรรพ์ครั้นฟื้นองค์ดำ<r>[a][n]รันดร์</r>	จึงสั่ง<r>[a][n]กัลป์</r>กุมารอ่อนฤท<r>ัย</r>
ไปเชิญองค์นางนรัง<r>[a][j]ไข</r>	กลับมา<r>[a][j]ให้</r>พร้อมพรูพระผู้<r>[a][n]พัน</r>
นางผีเสื้อเสด็จไปใน<r>[a][N]กลาง</r>	พบกุ<r>[a][N]มาร</r>ก็ขึ้นประทับ<r>[a][n]นัสน์</r>
ไม่ขัดข้องหมองทำไม่จำ<r>[a][n]กัน</r>	พระกุ<r>[a][n]มาร</r>เห็นนางเข้าคลาน<r>[a]มา</r>
ช่างดีงามหลากจิตพิศ<r>[a][t]วาส</r>	จะ<r>[a][t]ขัด</r>ขวางคงมีไม่ส<r>[a]มา</r>
อย่าไปเลยขืนทูลจงผิน<r>[a]หน้า</r>	อยู่<r>[a]ท่า</r>ไปเถิดจะไปเฝ้าพระ<r>[o][N]องค์</r>
กษัตริย์ให้เข้าในหอนรา<r>[o][p]ณพ</r>	ยังจะ<r>[o][p]หลบ</r>หลีกลับไม่กลับ<r>[o][N]หลง</r>
พระโฉมยงทรงฟังก็ยัง<r>[o][N]สง</r>	สั่งข้า<r>[o][N]คง</r>ไปให้เป็นสำ<r>[a][n]คัญ</r>
เมื่อเช้าตรู่จู่มามิขัด<r>[a][j]ใจ</r>	เราจะ<r>[a][j]ได้</r>ชมนางข้างอัม<r>[a][n]พันธุ์</r>
นางเสด็จขึ้นบนยอดมณฑาส<r>[a][n]ถาน</r>	ครั้นแจ้ง<r>[a][n]การ</r>กัลป์กุมาคี<r>[i]รี</r>
สั่งให้เตรียมเครื่องประพาส<r>[ua][N]หลวง</r>	พระอ<r>[ua][N]นร</r>กษัตริย์กับอัค<r>[i]นี</r>
ที่ในประเทศสัตนาถึงธา<r>[

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [35]:
hub_model_id= f"Pongsaky/{experiment_name}"
hub_model_id

'Pongsaky/gemma-3-4b-full-training'

In [36]:
model.save_pretrained(experiment_name)  # Local saving
tokenizer.save_pretrained(experiment_name)
model.push_to_hub(hub_model_id, token = HUGGINGFACE_API_KEY) # Online saving
tokenizer.push_to_hub(hub_model_id, token = HUGGINGFACE_API_KEY) # Online saving

/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:220: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


README.md:   0%|          | 0.00/599 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Saved model to https://huggingface.co/Pongsaky/gemma-3-4b-full-training


  0%|          | 0/2 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
base_model, tokenizer_lora = FastLanguageModel.from_pretrained(
      model_name="scb10x/llama3.2-typhoon2-1b",
      max_seq_length=1024,
      load_in_4bit=False,
  )

  # Add new token
  add_new_tokens(base_model, tokenizer_lora, phonetic_token)

  # Load LoRA adapter
  peft_model_id = "Pongsaky/llama3.2-typhoon2-1b-lora-unfreeze-embedding-phonetic"
  model_lora = PeftModel.from_pretrained(base_model, peft_model_id)

  # Enable native 2x faster inference
  FastLanguageModel.for_inference(model_full)

In [16]:
import gc

# Clear deleted GPU items
for _ in range(3):
    gc.collect()
    torch.cuda.empty_cache()

In [17]:
from peft import PeftModel
from unsloth import FastLanguageModel, FastModel

base_model, base_tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-pt-unsloth-bnb-4bit",
    # model_name = "./gemma-3-4b-full-training",
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    token = HUGGINGFACE_API_KEY, # use one if using gated models like meta-llama/Llama-2-7b-hf,
)

add_new_tokens(base_model, base_tokenizer, tag_dict, new_tokens=phonetic_token)

# # Load LoRA adapter
peft_model_id = "Pongsaky/gemma-3-4b-full-training"
model_lora = PeftModel.from_pretrained(base_model, peft_model_id)

FastLanguageModel.for_inference(model_lora)
# loaded_model, loaded_tokenizer = FastModel.from_pretrained("Pongsaky/gemma-3-4b-full-training")

==((====))==  Unsloth 2025.3.19: Fast Gemma3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.673 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: You're adding new_tokens = ['<klon8>', '<phonetic>', '<th>', '<r>', '</r>', '[0]', '[1]', '[2]', '[3]', '[4]', '[?]', '[@@]', '[N]', '[OO]', '[O]', '[UU]', '[UUa]', '[U]', '[a]', '[aa]', '[b]', '[c]', '[ch]', '[d]', '[e]', '[ee]', '[f]', '[h]', '[i]', '[ii]', '[iia]', '[j]', '[k]', '[kh]', '[khl]', '[khr]', '[khw]', '[kl]', '[kr]', '[kw]', '[l]', '[m]', '[n]', '[o]', '[oo]', '[p]', '[ph]', '[phl]', '[phr]', '[pl]', '[pr]', '[r]', '[s]', '[sr]', '[t]', '[th]', '[thr]', '[tr]', '[u]', '[uu]', '[uua]', '[w]', '[x]', '[xx]']

/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:550: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['language_model.model.embed_tokens', 'language_model.lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (vision_tower): SiglipVisionModel(
        (vision_model): SiglipVisionTransformer(
          (embeddings): SiglipVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
            (position_embedding): Embedding(4096, 1152)
          )
          (encoder): SiglipEncoder(
            (layers): ModuleList(
              (0-26): 27 x SiglipEncoderLayer(
                (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                (self_attn): SiglipAttention(
                  (k_proj): lora.Linear(
                    (base_layer): Linear(in_features=1152, out_features=1152, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Identity()
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=1152, out_fe

In [19]:
model_lora.merge_and_unload()

/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:392: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['language_model.model.embed_tokens', 'language_model.lm_head'] are part of the adapter. This can lead to complications. You can opt to merge the adapter after cloning the weights (to untie the embeddings). You can untie the embeddings by loading the model with `tie_word_embeddings=False`. For example:
```python
from transformers import AutoModelForCausalLM

# Load original tied model
model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it", tie_word_embeddings=False)

# Set the randomly initialized lm_head to the previously tied embeddings
model.lm_head.weight.data = model.model.embed_tokens.weight.data.clone()

# Save the untied model
untied_model_dir = "dir/for/untied/model"
model.save_pretrained(untied_model_dir)
model.config.save_pretrained(untied_model_dir)

# Now use the original model but in untied format
model = Auto

Gemma3ForConditionalGeneration(
  (vision_tower): SiglipVisionModel(
    (vision_model): SiglipVisionTransformer(
      (embeddings): SiglipVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(4096, 1152)
      )
      (encoder): SiglipEncoder(
        (layers): ModuleList(
          (0-26): 27 x SiglipEncoderLayer(
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (self_attn): SiglipAttention(
              (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (mlp): SiglipMLP(
            

In [ ]:
text = "<bos><klon8> คนธรรพ์ครั้น"

from transformers import TextStreamer
_ = model_lora.generate(
    **base_tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 128, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(base_tokenizer, skip_prompt = False),
)

<bos><klon8> 

In [ ]:
if True:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "Pongsaky/gemma-3-4b-it-unsloth-bnb-4bit-phonetic-fine-tuned", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

text = "<bos><klon8> พอได้ยินเสียงระฆัง"

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

==((====))==  Unsloth 2025.3.19: Fast Gemma3 patching. Transformers: 4.50.0.dev0. vLLM: 0.8.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


adapter_model.safetensors:   0%|          | 0.00/59.7M [00:00<?, ?B/s]

<bos><klon8> พอได้ยินเสียงระฆังดังไหวไหว
เสียงก้องกังวานแว่วแว่วไปในไพร
เสียงระฆังดังไหวเหมือนไหวไหว
เสียงก้องกังวานแว่วแว่วไปในไพร
เสียงระฆังดังไหวเหมือนไ


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-3-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-3-finetune", tokenizer,
        token = "hf_..."
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # Only Q8_0, BF16, F16 supported
        repo_id = "HF_ACCOUNT/gemma-finetune-gguf",
        token = "hf_...",
    )

Now, use the `gemma-3-finetune.gguf` file or `gemma-3-finetune-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
